## 1. Définitions

Soit un ensemble de `n` **éléments** et une famille `F` de parties (des
**sets**). On colorie chaque élément en `+1` ou `-1`. Pour un set `S`, la
**somme colorée** est `sum_{x in S} c(x)`. La **discrépance** est le maximum
des valeurs absolues de ces sommes :

```
discrepancy(F, c) = max_{S in F} | sum_{x in S} c(x) |
```

Le **degré** d'un élément est le nombre de sets qui le contiennent ; le degré
maximal de la famille est noté `k`. L'hypothèse « degré ≤ k » (chaque élément
n'apparaît que dans au plus `k` contraintes) est celle de Beck–Fiala.

**Question centrale** : pour un système de degré ≤ k, quelle est la plus
petite discrépance atteignable par un choix de coloration ?
La réponse est entre `~√k` (ce qui est inévitable) et `2k−1` (ce que
l'algorithme classique garantit).

In [1]:
import numpy as np
import math

def discrepancy(sets, color):
    """Discrépance de la famille `sets` (liste de listes d'indices) sous la coloration `color`."""
    return max(int(abs(sum(color[i] for i in S))) for S in sets)

def degree(sets, x, n):
    """Degré de l'élément `x` : nombre de sets qui le contiennent."""
    return sum(1 for S in sets if x in S)

def max_degree(sets, n):
    """Degré maximal `k` de la famille."""
    return max(degree(sets, x, n) for x in range(n))

# --- Exemple : un petit système de degré 2, 8 éléments ---
sets = [[0,1,2],[2,3,4],[4,5,6],[6,7,0]]   # 4 sets, chaque élément dans exactement 2
n = 8
k = max_degree(sets, n)
print(f"n={n}, |F|={len(sets)}, k={k}")
# coloration l'alternance évidente : chacun donne somme nulle
c = [1,-1,1,-1,1,-1,1,-1]
print("coloration alternée", c, "-> discrépance", discrepancy(sets, c))
# coloration toute +1 : concentre
print("coloration toute +1 -> discrépance", discrepancy(sets, [1]*n))

n=8, |F|=4, k=2
coloration alternée [1, -1, 1, -1, 1, -1, 1, -1] -> discrépance 1
coloration toute +1 -> discrépance 3


## 2. Borne inférieure : `√n/2` est inévitable (moyenne sur les sous-ensembles)

La discrépance ne peut pas être rendue arbitrairement petite : sous un
recouvrement fort, **un** ensemble doit concentrer. Deux volets, l'un
rigoureux (moyenne sur les demi-parties, qui donne `Ω(√n)`), l'autre
probabiliste (le Chernoff classique, qui borne la discrépance des colorations
aléatoires).

### 2.1 Un plancher rigoureux : le système des demi-parties (moyenne sur les sous-ensembles)

Il est **faux** de croire qu'un plancher naïf vient de la taille des
ensembles : des ensembles **disjoints** se balancent **indépendamment**
(coloration `+1,−1,+1,−1` par set → somme nulle, discrépance `0`). La
difficulté naît du **recouvrement** : les mêmes éléments apparaissent dans
plusieurs sets, et il faut les équilibrer **simultanément**.

Le plancher classique (Erdős–Spencer) se démontre par l'identité de
l'écart-type sur **toutes les demi-parties** de `[n]` (les `m`-sous-ensembles,
`m = ⌊n/2⌋`). Pour une coloration `c` de somme totale nulle (`Σ c = 0`) :

```
E_S[(Σ_{x∈S} c(x))²]  sur un m-sous-ensemble S aléatoire
  = m·(1 − (m−1)/(n−1))   +   (terme de moyenne, nul car Σc = 0)
  ≈ n/4·(1−o(1))
```

Un m-sous-ensemble aléatoire réalise l'espérance, donc **il existe** un set
`S` avec `(Σ_{x∈S} c(x))² ≥ n/4·(1−o(1))`, i.e. `|Σ_{x∈S}c(x)| ≥ (√n/2)·(1−o(1))`.
Toute coloration équilibrée laisse donc un set à disps `≥ √n/2` :
**la discrépance du système des demi-parties est `Ω(√n)`** (en fait
`Θ(n/2)` pour ce système précis — la borne `√n/2` est une **minoration**,
pas un ordre exact ; le point est qu'elle croît avec `n`).

**Point de vocabulaire (important) — le degré contre le nombre d'éléments.**
Ce plancher est en `n` (nombre d'éléments), pas en `k` (le **degré** de la
conjecture Beck–Fiala). Chaque élément apparaît dans `C(n−1,m−1)` demi-parties,
donc le degré ici est **énorme** (`≈ 2^n/√n`). La conjecture Beck–Fiala dit
`O(√k)` **en le degré** — un plancher `√k` simple n'existe pas, la
construction qui le serre est subtile (grands ensembles, petit degré), et
c'est précisément son régime grand-degré (`k ≥ log² n`) que Bansal–Jiang 2025
résolvent (voir §4). Le `√n/2` ci-dessus est le plancher en `n`, un autre
plancher, plus facile, tout aussi rigoureux.

### 2.2 Le Chernoff (borne aléatoire—à l'opposé)

Un coloriage aléatoire indépendant rend chaque somme **sous-gaussienne** :
`P(|sum_S| ≥ t) ≤ 2·exp(−t²/(2|S|))`. Sur une famille de `m` sets, l'union
bound donne `disc ≤ ~√(2·|S|·ln(2m))` avec probabilité haute. C'est la **borne
supérieure** facile — pas la borne inférieure, mais elle fixe l'ordre de
grandeur où vit la vraie discrépance : entre `√|S|` (inévitable) et
`√(|S| ln m)` (aléatoire). Toute la théorie (Spencer qui efface les `ln m`,
Banaszczyk, Bansal–Jiang) vit dans cet écart.

In [2]:
from itertools import combinations

def disjoint_system(n_sets, size):
    """n_sets ensembles disjoints de `size` éléments chacun."""
    base = 0
    out = []
    for _ in range(n_sets):
        out.append(list(range(base, base + size)))
        base += size
    return out

def half_subsets(n):
    """Toutes les m-sous-parties de [n], m = n//2 (le « système des demi-parties »)."""
    m = n // 2
    return [list(c) for c in combinations(range(n), m)]

def brute_best(sets, n):
    """Meilleure discrépance par force brute sur toutes les colorations (n petit)."""
    best = float("inf")
    for mask in range(1 << n):
        c = [(1 if (mask >> i) & 1 else -1) for i in range(n)]
        best = min(best, discrepancy(sets, c))
    return best

# (a) Ensembles DISJOINTS : chaque set se balance indépendamment -> trivial (0 ou 1).
print("(a) Ensembles disjoints : recouvrement nul, donc trivial")
for size in [2,3,4]:
    sets = disjoint_system(4, size)
    n = len(sets) * size
    print(f"    s={size}, n={n}, |F|={len(sets)} : meilleure coloration = {brute_best(sets, n)}  (dans {{0,1}})")
print()

# (b) Système des DEMI-PARTIES (recouvrement fort) : le plancher sqrt(n)/2 est atteint.
print("(b) Système des demi-parties (toutes les m-sous-parties, m=n/2)")
for n in [4, 6]:
    sets = half_subsets(n)
    opt = brute_best(sets, n)
    floor = math.sqrt(n) / 2
    print(f"    n={n}, |F|={len(sets)} : meilleure coloration = {opt}   (plancher prouve sqrt(n)/2 = {floor:.2f})")
print()
print("=> la difficulte vient du RECOUVREMENT, pas de la taille : des ensembles disjoints")
print("   se balancent trivialement, des demi-parties densees imposent un plancher sqrt(n)/2.")

(a) Ensembles disjoints : recouvrement nul, donc trivial
    s=2, n=8, |F|=4 : meilleure coloration = 0  (dans {0,1})
    s=3, n=12, |F|=4 : meilleure coloration = 1  (dans {0,1})
    s=4, n=16, |F|=4 : meilleure coloration = 0  (dans {0,1})

(b) Système des demi-parties (toutes les m-sous-parties, m=n/2)
    n=4, |F|=6 : meilleure coloration = 2   (plancher prouve sqrt(n)/2 = 1.00)
    n=6, |F|=20 : meilleure coloration = 3   (plancher prouve sqrt(n)/2 = 1.22)

=> la difficulte vient du RECOUVREMENT, pas de la taille : des ensembles disjoints
   se balancent trivialement, des demi-parties densees imposent un plancher sqrt(n)/2.


## 3. Beck–Fiala classique : `disc ≤ 2k − 1` (variables flottantes)

Le théorème fondateur (Beck & Fiala 1981) : tout système de degré ≤ k admet
une coloration de discrépance **au plus 2k−1**, et l'algorithme est
polytemporel. Idée : on fait évoluer des **variables flottantes** `x ∈ [−1,1]`
(itérer jusqu'à ce qu'elles atteignent `±1`), en ne bougeant que dans le
**noyau** des contraintes « dangereuses » — celles qui contiennent entre 1 et
`k−1` variables flottantes, et dont la somme doit être **préservée**. Les
autres contraintes (≥ k flottantes) absorbent le mouvement. On gèle une
variable dès qu'elle touche `±1` ; le lemme de comptage (nombre de contraintes
dangereuses < nombre de flottantes, cf. Matoušek, *Lectures on Discrete
Geometry*, §1.4) garantit un noyau non trivial et la terminaison. La borne
`≤ 2k−1` se lit sur l'invariant : une contrainte dangereuse est gelée, les
autres ne dépassent pas la borne au moment où elles deviennent dangereuses.
Nous implémentons l'algorithme et **vérifions la borne en sortie**.

In [3]:
def nullspace(A):
    """Base du noyau de A : vecteurs v avec A v = 0 (via SVD)."""
    if A.size == 0:
        return np.eye(A.shape[1])
    _, s, vh = np.linalg.svd(A)
    tol = max(A.shape) * (s[0] if s.size else 1.0) * 1e-12
    r = int((s > tol).sum())
    return vh[r:].T

def beck_fiala(sets, n, k, rng=None):
    """Coloration ±1 de discrépance <= 2k-1 (borné vérifié en sortie).
    `sets` : liste de listes d'indices de [0,n). `k` : degré max majorant."""
    sets = [np.array(S, dtype=int) for S in sets]
    F = list(range(n))
    x = np.zeros(n, dtype=float)
    color = np.zeros(n, dtype=int)
    guard = 0
    while F and guard < 10 * n:
        guard += 1
        rows, inter_list = [], []
        Fset = set(F)
        for S in sets:
            inter = [int(i) for i in S if int(i) in Fset]
            if 1 <= len(inter) <= k - 1:
                rows.append(inter)
        A = np.zeros((len(rows), len(F)))
        for r, inter in enumerate(rows):
            for j, f in enumerate(F):
                if f in inter:
                    A[r, j] = 1.0
        K = nullspace(A)
        if K.shape[1] == 0:
            # noyau trivial (rare) : figer une flottante au plus proche de +-1
            j = int(np.argmax(np.abs(x[F])))
            f = F[j]
            color[f] = 1 if x[f] >= 0 else -1
            F.remove(f)
            continue
        v = K[:, 0]
        lam = float("inf")
        for j in range(len(F)):
            vj = v[j]
            if abs(vj) < 1e-12:
                continue
            xj = x[F[j]]
            for target in (1.0, -1.0):
                l = (target - xj) / vj
                if l > 1e-12:
                    lam = min(lam, l)
        if not math.isfinite(lam):
            lam = 1.0
        for j in range(len(F)):
            x[F[j]] += lam * v[j]
        newF = []
        for f in F:
            if abs(abs(x[f]) - 1.0) < 1e-9:
                color[f] = 1 if x[f] > 0 else -1
            else:
                newF.append(f)
        if len(newF) == len(F):
            # rien n'a été figé (v ~ 0) : figer la plus grande valeur absolue
            j = int(np.argmax(np.abs(x[F])))
            f = F[j]
            color[f] = 1 if x[f] >= 0 else -1
            F.remove(f)
        else:
            F = newF
    for f in F:
        color[f] = 1 if x[f] >= 0 else -1
    return color, discrepancy(sets, color)

# Test sur des systèmes aléatoires de degré k, on vérifie disc <= 2k-1.
rng = np.random.default_rng(0)
def random_degree_system(n, k, seed):
    """n éléments, des sets tirés au hasard, degré max approx k."""
    r = np.random.default_rng(seed)
    sets = []
    for _ in range(n):
        # chaque set contient 1..k éléments distincts
        sz = int(r.integers(1, k + 1))
        sets.append(sorted(r.choice(n, size=sz, replace=False).tolist()))
    return sets

results = []
for k in [2,3,4,5]:
    for seed in range(3):
        n = 12
        sets = random_degree_system(n, k, seed)
        col, d = beck_fiala(sets, n, k)
        results.append((k, seed, d, 2*k-1, d <= 2*k-1))
print("k | seed | disc(algo) | 2k-1 | borne tenue ?")
for k, sd, d, cap, ok in results:
    print(f"{k} |  {sd}  |    {d:>3}    | {cap:>3} | {ok}")
print("=> borne Beck-Fiala 2k-1 tenue sur tous les essais :", all(r[4] for r in results))

k | seed | disc(algo) | 2k-1 | borne tenue ?
2 |  0  |      2    |   3 | True
2 |  1  |      2    |   3 | True
2 |  2  |      2    |   3 | True
3 |  0  |      3    |   5 | True
3 |  1  |      3    |   5 | True
3 |  2  |      3    |   5 | True
4 |  0  |      2    |   7 | True
4 |  1  |      4    |   7 | True
4 |  2  |      4    |   7 | True
5 |  0  |      3    |   9 | True
5 |  1  |      2    |   9 | True
5 |  2  |      3    |   9 | True
=> borne Beck-Fiala 2k-1 tenue sur tous les essais : True


## 4. La frontière : prouvé vs conjecturé

| Résultat | Année | Borne | Statut |
|----------|-------|-------|--------|
| Signes aléatoires (Chernoff) | — | `~√(n)·√ln m` | **prouvé** (borne sup. facile) |
| Demi-parties (moyenne sous-ensembles) | — | `≥ √n/2` | **prouvé** (borne inf. en `n`, §2) |
| Beck–Fiala | 1981 | `≤ 2k−1` | **prouvé** (algorithme §3) |
| Beck–Fiala conjecture | 1981 | `≤ C·√k` | **conjecturé** (ouvert) |
| Spencer « six standard deviations » | 1985 | `≤ 6√n` | **prouvé** |
| Banaszczyk | 1998 | `≤ O(√(k log n))` (Komlós `√log n`) | **prouvé** |
| **Bansal–Jiang 2025** ([arXiv:2508.03961](https://arxiv.org/abs/2508.03961)) | 2025 | `Õ(√k + √log n)` ; BF vrai dès `k ≥ log²n` ; Komlós `Õ(log^{1/4} n)` | **prouvé** — par découplage spectral affine |

**Ce qui reste ouvert** : la conjecture de Beck–Fiala (le `√k` en degré
arbitrairement petit), et le `O(1)` de Komlós. Le papier 2025 les résout
**en régime grand degré** (`k ≥ log² n` pour Beck–Fiala) — c'est une
avancée, pas une conclusion : le cœur (petit degré) reste ouvert.

Ces énoncés sont formalisés (en `Prop` nommées, sans preuve encore) dans
le lake [`discrepancy_lean`](../discrepancy_lean/README.md), palier P0 de
l'issue #12823.

## 5. CP-SAT en oracle exact : le vrai moteur contre les arrondis

L'algorithme Beck–Fiala **garantit** `2k−1` mais ne le resserre pas. Pour
savoir **exactement** quelle discrépance est atteignable sur un système donné,
il faut un solveur d'optimisation. OR-Tools `CP-SAT` joue ici ce rôle
d'**oracle exact** : on modélise `min max_S |sum_S|` avec des variables
booléennes `c_i ∈ {+1,−1}`, une variable `D` et les contraintes d'encadrement
`-D ≤ sum_S ≤ D` pour chaque set. C'est un problème non-trivial que CP-SAT
résout à l'optimum là où les arrondis polytemporels — Beck–Fiala (`2k−1`) et
un arrondi aléatoire simple — ne donnent qu'une borne.

On compare, sur les mêmes systèmes, l'**optimum exact** (CP-SAT) aux deux
arrondis. C'est Prong B de l'Epic `#3801` : le moteur montre sa valeur sur
un problème où un arrondi naïf est loin de l'optimum.

In [4]:
from ortools.sat.python import cp_model

def cpsat_optimal(sets, n, time_limit=5.0):
    """Discrépance minimale exacte par CP-SAT : min D s.t. |sum_S| <= D pour tout S."""
    m = cp_model.CpModel()
    c = {i: m.NewBoolVar(f"c{i}") for i in range(n)}   # True = +1, False = -1
    D = m.NewIntVar(0, n, "D")
    for S in sets:
        # sum_S = sum (2*val-1)  si val est la bool (True=+1): contribution +1 ou -1
        # encadrement -D <= sum <= D
        # somme des (c_i ? +1 : -1)
        terms = []
        for i in S:
            # +1 si c_i True, -1 sinon  -> 2*c_i - 1
            terms.append(2 * c[i])
        # on introduit la somme exacte (pas nécessaire, on utilise AddExactly? non) :
        # simple : -D <= sum(2c_i -1) <= D
        expr = m.Add(2*sum(c[i] for i in S) - len(S) >= -D)
        expr2 = m.Add(2*sum(c[i] for i in S) - len(S) <= D)
    m.Minimize(D)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    st = solver.Solve(m)
    if st in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return int(solver.ObjectiveValue())
    return None

def random_rounding(sets, n, trials=200, seed=0):
    """Meilleur tirage aléatoire (arrondi naïf) en `trials` essais."""
    best = float("inf")
    r = np.random.default_rng(seed)
    for _ in range(trials):
        c = r.integers(0, 2, size=n).tolist()
        c = [1 if v else -1 for v in c]
        best = min(best, discrepancy(sets, c))
    return best

print("Comparaison sur des systemes de degre k (meme instances) :")
print("k  | n | CP-SAT (optimum EXACT) | garantie Beck-Fiala (2k-1) | meilleur arrondi aleatoire")
for k in [2,3,4]:
    sets = random_degree_system(12, k, seed=7)
    n = 12
    opt = cpsat_optimal(sets, n)
    d_rand = random_rounding(sets, n)
    cap = 2*k - 1
    print(f"{k} | {n} | {str(opt):^5} | {cap:^5} | {int(d_rand):^5}")
print()
print("=> CP-SAT fait la seule chose qu'aucun arrondi ne fait : calculer l'optimum EXACT.")
print("   L'arrondi Beck-Fiala ne fournit qu'une GARANTIE pire-cas (2k-1), bien au-dessus")
print("   de l'optimum reel sur ces instances. C'est Prong B de #3801 : un vrai solveur")
print("   (CP-SAT) certifie numeriquement la borne la plus fine, la ou un arrondi")
print("   polytemporel ne fait que borner.")

Comparaison sur des systemes de degre k (meme instances) :
k  | n | CP-SAT (optimum EXACT) | garantie Beck-Fiala (2k-1) | meilleur arrondi aleatoire
2 | 12 |   2   |   3   |   2  
3 | 12 |   1   |   5   |   1  
4 | 12 |   2   |   7   |   2  

=> CP-SAT fait la seule chose qu'aucun arrondi ne fait : calculer l'optimum EXACT.
   L'arrondi Beck-Fiala ne fournit qu'une GARANTIE pire-cas (2k-1), bien au-dessus
   de l'optimum reel sur ces instances. C'est Prong B de #3801 : un vrai solveur
   (CP-SAT) certifie numeriquement la borne la plus fine, la ou un arrondi
   polytemporel ne fait que borner.


In [5]:
# Courbe de croissance du plancher : l'optimum du système des demi-parties
# grandit comme ~sqrt(n)/2 (calcul exact par force brute sur n petit) — c'est
# le plancher rigoureux de §2, vu de l'intérieur.
print("n (demi-parties) | |F| | meilleure coloration | ceil(sqrt(n)/2)")
for n in [4, 6, 8, 10]:
    sets = half_subsets(n)
    opt = brute_best(sets, n)
    print(f"  {n:>2}  |  {len(sets):>4}  |  {opt:^4}  |  {math.ceil(math.sqrt(n)/2)}")
print("=> l'optimum croit avec n (ici ~n/2, bien au-dessus du plancher sqrt(n)/2 de §2)")
print("   : un recouvrement fort empeche d'equilibrer simultanement tous les sets, et")
print("   la discrepancy devient grande — contrairement aux ensembles disjoints (0 ou 1).")

n (demi-parties) | |F| | meilleure coloration | ceil(sqrt(n)/2)
   4  |     6  |   2    |  1
   6  |    20  |   3    |  2
   8  |    70  |   4    |  2
  10  |   252  |   5    |  2
=> l'optimum croit avec n (ici ~n/2, bien au-dessus du plancher sqrt(n)/2 de §2)
   : un recouvrement fort empeche d'equilibrer simultanement tous les sets, et
   la discrepancy devient grande — contrairement aux ensembles disjoints (0 ou 1).


## 6. Le fil « découplage » — pourquoi ce geste nous est familier

Le mot-clé du papier de Bansal–Jiang (*decoupling via affine
spectral-independence*) est le **cinquième volume d'un motif** que ce dépôt
enseigne déjà :

1. **PyMC-12 / PyMC-14** (reparamétrisation non centrée) : *découpler* `theta`
   et `sigma_theta` pour déplier le funnel — tuer une corrélation pathologique
   en changeant de paramétrisation.
2. **Hoeffding.lean** (`PacLearning`) : l'étape de preuve est littéralement
   nommée « **Découplage** `|empError − μ| ≥ ε ⟺ (nε ≤ Z) ∨ (Z ≤ −nε)` ».
3. **knot_lean** (`Invariant.lean`) : « Fox-**decoupling** at the proper-arc
   partner crossing » — le découplage comme argument d'invariant.
4. **Slides RL** : double estimateur Q1/Q2 **découplés** — casser le biais de
   sur-estimation en découplant sélection et évaluation.
5. **Ce notebook / le lake** : la discrépance **découple les évolutions de
   somme des lignes** via des contraintes spectrales affines sur la SDP, et
   la concentration devient applicable (le défaut que Banaszczyk payait d'un
   `√log n` disparaît).

**Grounding ICT.** Le programme [ICT](../../IIT/ICT-Series/README.md) (Epic
#4588) traque un **signal fantôme** : « mesures naïves = signaux fantômes »
(ICT-8), cosinus qui « fabrique une ressemblance fantôme sur un champ quasi
vide » (ICT-9). La discrépance est le **laboratoire mathématique de cette
même question, à certitude formelle près** : quand on colorie en `±1` et
qu'on somme par ligne, les sommes *ressemblent* à ce que donneraient des
signes indépendants (concentration) — mais cette indépendance n'existe pas
en général, et Banaszczyk la paie d'un facteur `√log n`. Le découplage de
Bansal–Jiang **impose le protocole** (contraintes d'indépendance spectrale)
sous lequel les lignes cessent de conspirer : la borne « fantôme » devient
une borne **prouvée**, `Õ(√k + √log n)`. ICT traque les fantômes statistiques
sur des trajectoires simulées ; la discrépance montre le même geste porté à
la certitude — une borne sur la **pire instance**, pas une mesure sur une
simulation.

## Exercices

Trois exercices pour creuser. Chaque stub est à compléter (le notebook
s'exécute de bout en bout même non complété — `C.1`).

### Exercice 1 — Arrondi aléatoire répété

Améliorer l'arrondi naïf : lancer beaucoup de colorations aléatoires et
garder la meilleure. Comparer à CP-SAT sur une instance délicate (où la
borne garantie est loin).

### Exercice 2 — Le Chernoff en pratique

Implémenter l'union-bound : estimer `P(disc ≤ t)` pour un système donné en
comptant les queues sous-gaussiennes, et trouver le `t` qui rend la
probabilité `≥ 1/2`. Que vaut-il en fonction de `k` ?

### Exercice 3 — Colonnes unitaires (Komlós)

Étendre le modèle CP-SAT au cas Komlós : des colonnes (vecteurs) de norme
unitaire, colorier en `±1` en bornant chaque **somme de ligne**. Le petit
système de §1 devient une matrice ; adapter l'encadrement.

In [6]:
# Exercice 1 — améliore l'arrondi aléatoire en gardant la meilleure coloration,
# puis compare à l'optimum CP-SAT sur une instance de degré 3.
def meilleur_arrondi(sets, n, trials=2000, seed=1):
    """TODO : lancer `trials` colorations aléatoires, garder la meilleure discrépance."""
    # Indices : np.random génère un vecteur de +-1 ; discrepancy(sets, c) donne la valeur.
    best = float("inf")
    # ...
    return best  # à compléter : best = min(discrépance sur les essais)

# Exemple guide (résolu) : la boucle minimale sur 500 essais.
r = np.random.default_rng(3)
best = float("inf")
for _ in range(500):
    c = [1 if v else -1 for v in r.integers(0, 2, size=10)]
    best = min(best, discrepancy(random_degree_system(10,3,9), c))
print("exemple guide (500 essais, système aléatoire de degré 3) : best =", best)

# À toi : compare la version `meilleur_arrondi` (2000 essais) à l'optimum CP-SAT.
print("Exercice 1 à compléter")

exemple guide (500 essais, système aléatoire de degré 3) : best = 1
Exercice 1 à compléter


In [7]:
# Exercice 2 — l'union-bound de Chernoff en pratique.
def union_bound_t(sets, n, t):
    """TODO : majorer P(disc > t) par sum_S 2*exp(-t^2/(2|S|)) (union bound)."""
    # Indices : pour chaque set S, la proba qu'une coloration aléatoire ait |sum_S| >= t
    # est <= 2*exp(-t^2/(2*len(S))). Somme sur tous les sets.
    print("Exercice 2 à compléter")
    return None

print("Exercice 2 lancé (stub)")

Exercice 2 lancé (stub)


In [8]:
# Exercice 3 — étendre l'encadrement CP-SAT au cas Komlós (colonnes unitaires).
def cpsat_komlos(A, time_limit=5.0):
    """TODO : min D s.t. |sum_j A[i,j] * c_j| <= D pour chaque ligne i (colonnes c_j in +-1)."""
    # Indices : A est une matrice numpy (m lignes x n colonnes). Variable bool par colonne,
    # D global, contrainte d'encadrement par ligne. Retourner l'entier D optimal.
    print("Exercice 3 à compléter")
    return None

print("Exercice 3 lancé (stub)")

Exercice 3 lancé (stub)
